In [2]:
import pandas as pd
import sys
sys.path.append('../Source')

from Pipeline.build_comparisons import buildComparisons

alignedData = pd.read_csv('../Data/Processed/alignedData.csv')
modelingData = buildComparisons(alignedData)

print(f'Shape: {modelingData.shape[0]} rows, {modelingData.shape[1]} columns')
modelingData.head()





Shape: 2346 rows, 5 columns


,week,regionI,regionJ,winsI,total
0,2018-01-01/2018-01-07,CaucasusCA,Europe,1050,1505
1,2018-01-01/2018-01-07,CaucasusCA,MiddleEast,1050,3660
2,2018-01-01/2018-01-07,CaucasusCA,NorthAfrica,1050,1320
3,2018-01-01/2018-01-07,Europe,MiddleEast,455,3065
4,2018-01-01/2018-01-07,Europe,NorthAfrica,455,725


In [3]:
import pandas as pd
import numpy as np
import arviz as az 

In [4]:
regions = ['CaucasusCA', 'Europe', 'MiddleEast', 'NorthAfrica']
regionMap = {'CaucasusCA' : 0, 'Europe' : 1, 'MiddleEast' : 2, 'NorthAfrica' : 3}

winsI = modelingData['winsI'].to_numpy()
totals = modelingData['total'].to_numpy()

indexI = modelingData['regionI'].map(regionMap).to_numpy()
indexJ = modelingData['regionJ'].map(regionMap).to_numpy()

nRegions = len(regions)

print(indexI[:10])
print(indexJ[:10])


[0 0 0 1 1 2 0 0 0 1]
[1 2 3 2 3 3 1 2 3 2]


In [5]:
import pymc as pm

with pm.Model() as model:
    # Establish Bayesian prior, constrained to sum to 0
    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nRegions)

    strengthI = beta[indexI]
    strengthJ = beta[indexJ]

    logOdds = strengthI - strengthJ
    probability = pm.math.sigmoid(logOdds)
     
    likelihood = pm.Binomial('likelihood', n=totals, p=probability, observed=winsI)

with model: 
    trace = pm.sample(1000, tune=1000, return_inferencedata=True)

Initializing NUTS using jitter+adapt_diag...
c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 38 seconds.


In [6]:
az.summary(trace, var_names=['beta'])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-0.686,0.001,-0.688,-0.685,0.0,0.0,4548.0,3072.0,1.0
beta[1],0.946,0.001,0.945,0.947,0.0,0.0,4260.0,3037.0,1.0
beta[2],0.849,0.001,0.848,0.850,0.0,0.0,4472.0,3026.0,1.0
beta[3],-1.109,0.001,-1.110,-1.107,0.0,0.0,2552.0,2767.0,1.0


In [7]:
samples = trace.posterior['beta'].values.reshape(-1, nRegions)

print(samples.shape)
print(samples[:5])

(4000, 4)
[[-0.6872507   0.947055    0.85000365 -1.10980795]
 [-0.68541113  0.94538405  0.84847538 -1.1084483 ]
 [-0.68684475  0.94625506  0.84998202 -1.10939233]
 [-0.68604762  0.9467028   0.84811826 -1.10877344]
 [-0.68577954  0.94554606  0.84954297 -1.1093095 ]]
